<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [14]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [15]:
type(docs)

list

In [16]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [ ]:
#comlete the code
import re
def preprocess(text):
    text = text.lower()                          
    text = re.sub(r'[^a-z\s]', '', text)         
    tokens = text.split()                        
    stopwords = {'the','is','in','it','and','to','a','of','for','on','are','with','that','this','was','as','at','by','an','be','from'}
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
    return tokens

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [ ]:
#comlete the code
def build_vocab(docs):
    vocab = set()
    for doc in docs:
        tokens = preprocess(doc)
        vocab.update(tokens)
    return sorted(vocab)   

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
def compute_df(docs, vocab):
    vocab_index = {w: i for i, w in enumerate(vocab)}
    df = [0] * len(vocab)
    for doc in docs:
        tokens = set(preprocess(doc))   
        for token in tokens:
            if token in vocab_index:
                df[vocab_index[token]] += 1
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
import math
def compute_idf(df, N):
    idf = []
    for d in df:
        # IDF: log((1+N)/(1+df)) + 1 
        idf.append(math.log((1 + N) / (1 + d)) + 1)
    return idf

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [21]:
def compute_tf(doc, vocab):
    vocab_index = {w: i for i, w in enumerate(vocab)}
    tokens = preprocess(doc)
    tf = [0.0] * len(vocab)
    for token in tokens:
        if token in vocab_index:
            tf[vocab_index[token]] += 1
    # Normalize by doc length
    total = sum(tf)
    if total > 0:
        tf = [v / total for v in tf]
    return tf

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [ ]:
import numpy as np

def build_tfidf(docs):
    vocab = build_vocab(docs)
    df = compute_df(docs, vocab)
    idf = compute_idf(df, len(docs))
    idf = np.array(idf)
    matrix = []
    for doc in docs:
        tf = np.array(compute_tf(doc, vocab))
        tfidf_vec = tf * idf       
        matrix.append(tfidf_vec)

    return np.array(matrix), vocab

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [ ]:
def cosine(a, b):
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [ ]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [25]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)

0.2068765757558786
I just recently bought a 4 MB ram card for my original mac portable 
(backlit) and have since had some bizarre crashes. It happens when I put 
the machine to sleep and wake the machine up. sometimes i
--------------------------------------------------
0.17528971269933918
I'm looking for a Singer Featherweight 221 sewing machine (old, black 
sewing machine in black case).

Please contact:
--------------------------------------------------
0.15060164100728773



The previous article referred to the fact that you could only use 20ns SIMMs in
a 50MHz machine, but that you could use 80ns SIMMs in slower machines. I just
pointed out that if you could only use 
--------------------------------------------------
0.1476762423263096

Yeah.  Sounds typical.  Windows makes all sorts of extra demands on hardware,
and therefore your machine can't keep up with things.  Ever notice how when
acessing the floppies in Windows, everything 
-----------------------------------------------